# Propaganda Detection - F1 & Accuracy Calculation Tool

This notebook computes evaluation metrics for propaganda span detection:
- **Span-level F1** (精确匹配 & 重叠匹配)
- **多标签分类 F1** (Macro/Micro/Weighted)
- **二分类 Accuracy & F1**
- **每个技术的详细指标**

---
**Usage:**
1. Update the file paths in the **Configuration** cell below
2. Run all cells
3. Inspect results and plots

**日期**: 2026-01-12  
**版本**: 1.0

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os
from pathlib import Path

# Root directory: should contain data_analy/, results/, technique_list/, etc.
BASE = Path("your/path/here")  # e.g. Path("/home/user/propaganda-span-detection")

# Language code for evaluation
# po = Polish | ru = Russian | en = English | bg = Bulgarian | sl = Slovenian
LANGUAGE_CODE = "po"

# Stage 1 method used to generate the prediction file
# Options: "voting_aggressiv", "context", "model"
ORIGINAL_MODE = "voting_aggressiv"

# Character-level localisation method
# Options: "ai", "rule", "combined"
Character_Level_Methods = "ai"


## 📦 Import libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    classification_report, confusion_matrix
)
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported")

## ⚙️ Configuration — update paths here

In [ ]:
# ==================== UPDATE THESE PATHS ====================
from pathlib import Path

LANGUAGE_CODE = "po"  # po=Polish, ru=Russian, bg=Bulgarian, sl=Slovenian
ORIGINAL_MODE = "voting_aggressiv" # Stage 1 method: 'voting_aggressiv', 'context', or 'model'
Character_Level_Methods = "ai" # Character-level method: 'ai', 'rule', or 'combined'
Base_DIR = Path(BASE)

# Prediction file path (TSV format)
# PREDICTION_FILE = f"your/path/here"
PREDICTION_FILE = Base_DIR / f'compare_result_F1/checkthat2024/debug_results/TCM/{LANGUAGE_CODE}_voting_aggressive_patched_converted.tsv'
# Gold-standard label file path (TXT format)
LABEL_FILE = Base_DIR / f"data_analy/overlap_analysis_results/{LANGUAGE_CODE}_overlap_articles/labels-subtask-3-spans.txt"


# PREDICTION_FILE = Base_DIR / 'compare_result_F1/checkthat2024/debug_results/ru_voting_aggressive_all_ai_results.tsv'
# LABEL_FILE      = Base_DIR / 'data_analy/overlap_analysis_results/ru_overlap_articles/labels-subtask-3-spans.txt'

# Column names in the prediction file (update if different)
PRED_COLUMNS = {
    'article_id': 'article_id',
    'technique': 'technique',
    'start': 'start',
    'end': 'end',
    'confidence': 'confidence'  # optional column
}


print("📝 Configuration:")
print(f"  Prediction file: {PREDICTION_FILE}")
print(f"  Label file: {LABEL_FILE}")

# 🔍 Preview article_id format to confirm prefix setting
print("\n🔍 article_id 格式预览（用于确认前缀设置）:")

try:
    _pred_df = pd.read_csv(PREDICTION_FILE, sep='\t', nrows=3)
    _pred_ids = _pred_df['article_id'].astype(str).tolist()
    print(f"  Prediction file — first 3 article_ids: {_pred_ids}")
    print(f"  → Suggested PRED_HAS_ARTICLE_PREFIX = {any(v.startswith('article') for v in _pred_ids)}")
except Exception as e:
    print(f"  ⚠️ Could not read prediction file: {e}")

try:
    _label_ids = []
    with open(LABEL_FILE, 'r', encoding='utf-8') as _f:
        for _line in _f:
            _parts = _line.strip().split('\t')
            if len(_parts) >= 4:
                _label_ids.append(_parts[0])
            if len(_label_ids) == 3:
                break
    print(f"  Label file — first 3 article_ids: {_label_ids}")
    print(f"  → Suggested LABEL_HAS_ARTICLE_PREFIX = {any(v.startswith('article') for v in _label_ids)}")
except Exception as e:
    print(f"  ⚠️ Could not read label file: {e}")


In [ ]:
# Does the prediction file use 'article' prefix in article_id?
PRED_HAS_ARTICLE_PREFIX = False  # True → 'article25106'  |  False → '25106'

# Does the label file use 'article' prefix in article_id?
LABEL_HAS_ARTICLE_PREFIX = True  # True → 'article25106'  |  False → '25106'

# Confidence threshold (0.0 = no filtering)
CONFIDENCE_THRESHOLD = 0.0  # 0.0 = disabled



print("📝 Configuration:")
print(f"  Prediction file: {PREDICTION_FILE}")
print(f"  Label file: {LABEL_FILE}")
print(f"  Confidence threshold: {CONFIDENCE_THRESHOLD}")

## 🔧 Helper functions

In [ ]:
def normalize_article_id(article_id, has_prefix):
    """
    Normalise article_id
    Convert 'article25106' or '25106' to bare numeric string '25106'
    """
    article_id = str(article_id)
    if has_prefix and article_id.startswith('article'):
        return article_id[7:]
    elif not has_prefix and article_id.startswith('article'):
        return article_id[7:]
    return article_id


def load_prediction_tsv(file_path, columns, has_prefix, threshold=0.0):
    """
    Load TSV prediction file
    """
    try:
        df = pd.read_csv(file_path, sep='\t', encoding='utf-8')
        print(f"\n✓ Prediction file loaded")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}")
        
        # Filter by confidence score
        if 'confidence' in df.columns and threshold > 0:
            original_len = len(df)
            df = df[df['confidence'] >= threshold]
            print(f"  ⚠️ Applying confidence threshold {threshold}: {original_len} -> {len(df)} spans")
        
        # Normalise article_id
        df['article_id_normalized'] = df[columns['article_id']].apply(
            lambda x: normalize_article_id(x, has_prefix)
        )
        
        return df
    except Exception as e:
        print(f"✗ Failed to load prediction file: {e}")
        return None


def load_label_txt(file_path, has_prefix):
    """
    Load TXT label file (gold standard)
    Format: article_id \t technique \t start \t end
    """
    try:
        spans = []
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    parts = line.split('\t')
                    if len(parts) >= 4:
                        article_id = normalize_article_id(parts[0], has_prefix)
                        technique = parts[1]
                        start = int(parts[2])
                        end = int(parts[3])
                        spans.append((article_id, technique, start, end))
        
        print(f"\n✓ Label file loaded")
        print(f"  Total spans: {len(spans)}")
        return spans
    except Exception as e:
        print(f"✗ Failed to load label file: {e}")
        return None


def convert_df_to_spans(df, columns):
    """
    Convert DataFrame to span list
    """
    spans = []
    for _, row in df.iterrows():
        article_id = row['article_id_normalized']
        technique = str(row[columns['technique']])
        start = int(row[columns['start']])
        end = int(row[columns['end']])
        spans.append((article_id, technique, start, end))
    
    print(f"\nConverted to spans: {len(spans)} 个")
    return spans


print("✅ Helper functions defined")

## 📊 Metrics calculator class

In [ ]:
class MetricsCalculator:
    """评估指标计算器"""
    
    def __init__(self):
        self.results = {}
    
    def calculate_span_f1(self, true_spans, pred_spans, match_type='exact'):
        """
        Compute span-level F1
        
        Args:
            true_spans: list of gold spans
            pred_spans: list of predicted spans
            match_type: 'exact' (full match) or 'overlap' (partial overlap)
        """
        print(f"\n{'='*70}")
        print(f"Span-level F1 ({match_type} match)")
        print('='*70)
        
        if match_type == 'exact':
            true_set = set(true_spans)
            pred_set = set(pred_spans)
            
            tp = len(true_set & pred_set)
            fp = len(pred_set - true_set)
            fn = len(true_set - pred_set)
            
        elif match_type == 'overlap':
            tp = 0
            matched_true = set()
            matched_pred = set()
            
            for i, pred in enumerate(pred_spans):
                pred_id, pred_tech, pred_start, pred_end = pred
                
                for j, true in enumerate(true_spans):
                    true_id, true_tech, true_start, true_end = true
                    
                    if pred_id == true_id and pred_tech == true_tech:
                        if not (pred_end <= true_start or pred_start >= true_end):
                            if j not in matched_true:
                                tp += 1
                                matched_true.add(j)
                                matched_pred.add(i)
                                break
            
            fp = len(pred_spans) - len(matched_pred)
            fn = len(true_spans) - len(matched_true)
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        results = {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'tp': tp,
            'fp': fp,
            'fn': fn
        }
        
        print(f"\nTP: {tp:,} | FP: {fp:,} | FN: {fn:,}")
        print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
        print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
        print(f"F1 Score:  {f1:.4f} ({f1*100:.2f}%)")
        
        self.results[f'span_{match_type}'] = results
        return results
    
    def calculate_multilabel_f1(self, true_spans, pred_spans):
        """
        Compute multi-label F1 (technique-level, position-agnostic)
        """
        print(f"\n{'='*70}")
        print("多标签分类 F1")
        print('='*70)
        
        # Group by article
        true_by_article = defaultdict(set)
        pred_by_article = defaultdict(set)
        
        for article_id, technique, _, _ in true_spans:
            true_by_article[article_id].add(technique)
        
        for article_id, technique, _, _ in pred_spans:
            pred_by_article[article_id].add(technique)
        
        all_articles = sorted(set(true_by_article.keys()) | set(pred_by_article.keys()))
        all_techniques = set()
        for techs in true_by_article.values():
            all_techniques.update(techs)
        for techs in pred_by_article.values():
            all_techniques.update(techs)
        all_techniques = sorted(list(all_techniques))
        
        # Build binary label matrix
        y_true = []
        y_pred = []
        
        for article_id in all_articles:
            true_techs = true_by_article.get(article_id, set())
            pred_techs = pred_by_article.get(article_id, set())
            
            true_vec = [1 if tech in true_techs else 0 for tech in all_techniques]
            pred_vec = [1 if tech in pred_techs else 0 for tech in all_techniques]
            
            y_true.append(true_vec)
            y_pred.append(pred_vec)
        
        # Compute aggregate metrics
        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
        weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        
        # Compute per-technique metrics
        tech_metrics = []
        for i, tech in enumerate(all_techniques):
            true_col = [row[i] for row in y_true]
            pred_col = [row[i] for row in y_pred]
            
            p = precision_score(true_col, pred_col, zero_division=0)
            r = recall_score(true_col, pred_col, zero_division=0)
            f1 = f1_score(true_col, pred_col, zero_division=0)
            support = sum(true_col)
            
            tech_metrics.append({
                'technique': tech,
                'precision': p,
                'recall': r,
                'f1': f1,
                'support': support
            })
        
        tech_df = pd.DataFrame(tech_metrics)
        tech_df = tech_df.sort_values('f1', ascending=False)
        
        print(f"\nArticles: {len(all_articles)} | Technique types: {len(all_techniques)}")
        print(f"\nMacro F1:    {macro_f1:.4f} ({macro_f1*100:.2f}%)")
        print(f"Micro F1:    {micro_f1:.4f} ({micro_f1*100:.2f}%)")
        print(f"Weighted F1: {weighted_f1:.4f} ({weighted_f1*100:.2f}%)")
        
        results = {
            'macro_f1': macro_f1,
            'micro_f1': micro_f1,
            'weighted_f1': weighted_f1,
            'tech_metrics': tech_df,
            'all_techniques': all_techniques
        }
        
        self.results['multilabel'] = results
        return results
    
    def calculate_binary_accuracy(self, true_spans, pred_spans):
        """
        Compute binary classification accuracy (article level: propaganda vs. clean)
        """
        print(f"\n{'='*70}")
        print("二分类准确率（文章级别）")
        print('='*70)
        
        # Collect all article IDs
        true_articles = set(span[0] for span in true_spans)
        pred_articles = set(span[0] for span in pred_spans)
        all_articles = true_articles | pred_articles
        
        # Build binary labels
        y_true = [1 if aid in true_articles else 0 for aid in sorted(all_articles)]
        y_pred = [1 if aid in pred_articles else 0 for aid in sorted(all_articles)]
        
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        
        print(f"\n总Articles: {len(all_articles)}")
        print(f"Propaganda articles (gold): {sum(y_true)}")
        print(f"Propaganda articles (predicted): {sum(y_pred)}")
        print(f"\nAccuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
        print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
        print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
        print(f"F1 Score:  {f1:.4f} ({f1*100:.2f}%)")
        
        results = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'y_true': y_true,
            'y_pred': y_pred
        }
        
        self.results['binary'] = results
        return results


print("✅ Metrics calculator class defined")

## 📥 Load data

In [ ]:
print("="*70)
print("Loading data")
print("="*70)

# 加载预测文件
pred_df = load_prediction_tsv(
    PREDICTION_FILE, 
    PRED_COLUMNS, 
    PRED_HAS_ARTICLE_PREFIX,
    CONFIDENCE_THRESHOLD
)

# 加载标签文件
true_spans = load_label_txt(LABEL_FILE, LABEL_HAS_ARTICLE_PREFIX)

# 转换预测为spans
pred_spans = convert_df_to_spans(pred_df, PRED_COLUMNS)

# Alignment check
pred_articles = set(span[0] for span in pred_spans)
true_articles = set(span[0] for span in true_spans)

print(f"\n{'='*70}")
print("Alignment check")
print('='*70)
print(f"Predicted spans: {len(pred_spans):,}")
print(f"Gold spans: {len(true_spans):,}")
print(f"Predicted articles: {len(pred_articles)}")
print(f"Gold articles: {len(true_articles)}")
print(f"Overlapping articles: {len(pred_articles & true_articles)}")

if pred_articles - true_articles:
    diff = list(pred_articles - true_articles)[:5]
    print(f"⚠️ Articles only in predictions: {diff}...")
if true_articles - pred_articles:
    diff = list(true_articles - pred_articles)[:5]
    print(f"⚠️ Articles only in gold labels: {diff}...")

## 🧮 Run all metrics

In [ ]:
# Instantiate calculator
calculator = MetricsCalculator()

# # 1. Span-level F1 (exact match)
# exact_results = calculator.calculate_span_f1(true_spans, pred_spans, match_type='exact')

# 2. Span-level F1 (overlap match)
overlap_results = calculator.calculate_span_f1(true_spans, pred_spans, match_type='overlap')

# 3. Multi-label classification F1
multilabel_results = calculator.calculate_multilabel_f1(true_spans, pred_spans)

# 4. Binary classification accuracy
binary_results = calculator.calculate_binary_accuracy(true_spans, pred_spans)

## 📊 Per-technique metrics

In [ ]:
# Display per-technique metrics
tech_df = multilabel_results['tech_metrics']

print("\n" + "="*100)
print("Performance by propaganda technique")
print("="*100)

# Pandas display options
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 40)

# Formatted output
display_df = tech_df.copy()
display_df['precision'] = display_df['precision'].apply(lambda x: f"{x:.3f}")
display_df['recall'] = display_df['recall'].apply(lambda x: f"{x:.3f}")
display_df['f1'] = display_df['f1'].apply(lambda x: f"{x:.3f}")

print(display_df.to_string(index=False))

# Count techniques per F1 range
print("\n" + "="*100)
print("F1 score distribution")
print("="*100)

f1_ranges = [
    (0.0, 0.3, "poor"),
    (0.3, 0.5, "below average"),
    (0.5, 0.7, "moderate"),
    (0.7, 0.9, "good"),
    (0.9, 1.0, "excellent")
]

for low, high, label in f1_ranges:
    count = len(tech_df[(tech_df['f1'] >= low) & (tech_df['f1'] < high)])
    print(f"F1 [{low:.1f}, {high:.1f}): {count:2d} 个技术 - {label}")

## 📋 Error analysis

In [ ]:
print("\n" + "="*70)
print("错误分析")
print("="*70)

# Compute FP and FN per technique
true_set = set(true_spans)
pred_set = set(pred_spans)

fp_spans = pred_set - true_set
fn_spans = true_set - pred_set

fp_by_tech = Counter([span[1] for span in fp_spans])
fn_by_tech = Counter([span[1] for span in fn_spans])

# Top 10 false-positive techniques
print("\nTechniques with most false positives (top 10):")
print("-" * 70)
print(f"{'Technique':<40} {'Count':>10} {'Percentage':>15}")
print("-" * 70)
for tech, count in fp_by_tech.most_common(10):
    pct = count / len(fp_spans) * 100 if fp_spans else 0
    print(f"{tech:<40} {count:>10,} {pct:>14.1f}%")

# Top 10 false-negative techniques
print("\nTechniques with most false negatives (top 10):")
print("-" * 70)
print(f"{'Technique':<40} {'Count':>10} {'Percentage':>15}")
print("-" * 70)
for tech, count in fn_by_tech.most_common(10):
    pct = count / len(fn_spans) * 100 if fn_spans else 0
    print(f"{tech:<40} {count:>10,} {pct:>14.1f}%")

## 📈 Visualisation

In [ ]:
# # 1. Overview bar chart
# fig, axes = plt.subplots(2, 2, figsize=(15, 10))
# fig.suptitle('Propaganda Detection Performance Overview', fontsize=16, fontweight='bold')

# # Subplot 1: Span-level F1对比
# ax1 = axes[0, 0]
# metrics_names = ['Exact Match', 'Overlap Match']
# f1_scores = [exact_results['f1'], overlap_results['f1']]
# colors = ['#FF6B6B', '#4ECDC4']
# bars1 = ax1.bar(metrics_names, f1_scores, color=colors, alpha=0.8, edgecolor='black')
# ax1.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
# ax1.set_title('Span-level F1', fontsize=13, fontweight='bold')
# ax1.set_ylim([0, 1])
# ax1.grid(axis='y', alpha=0.3)
# for bar, score in zip(bars1, f1_scores):
#     height = bar.get_height()
#     ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
#              f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

# # Subplot 2: 多标签F1对比
# ax2 = axes[0, 1]
# ml_names = ['Macro', 'Micro', 'Weighted']
# ml_scores = [
#     multilabel_results['macro_f1'],
#     multilabel_results['micro_f1'],
#     multilabel_results['weighted_f1']
# ]
# colors2 = ['#FFD93D', '#6BCB77', '#4D96FF']
# bars2 = ax2.bar(ml_names, ml_scores, color=colors2, alpha=0.8, edgecolor='black')
# ax2.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
# ax2.set_title('Multi-label F1', fontsize=13, fontweight='bold')
# ax2.set_ylim([0, 1])
# ax2.grid(axis='y', alpha=0.3)
# for bar, score in zip(bars2, ml_scores):
#     height = bar.get_height()
#     ax2.text(bar.get_x() + bar.get_width()/2., height + 0.02,
#              f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

# # Subplot 3: 二分类性能
# ax3 = axes[1, 0]
# binary_names = ['Accuracy', 'Precision', 'Recall', 'F1']
# binary_scores = [
#     binary_results['accuracy'],
#     binary_results['precision'],
#     binary_results['recall'],
#     binary_results['f1']
# ]
# colors3 = ['#E63946', '#F77F00', '#06FFA5', '#118AB2']
# bars3 = ax3.bar(binary_names, binary_scores, color=colors3, alpha=0.8, edgecolor='black')
# ax3.set_ylabel('Score', fontsize=12, fontweight='bold')
# ax3.set_title('Binary Classification (Article-level)', fontsize=13, fontweight='bold')
# ax3.set_ylim([0, 1])
# ax3.grid(axis='y', alpha=0.3)
# for bar, score in zip(bars3, binary_scores):
#     height = bar.get_height()
#     ax3.text(bar.get_x() + bar.get_width()/2., height + 0.02,
#              f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

# # Subplot 4: TP/FP/FN对比
# ax4 = axes[1, 1]
# categories = ['True\nPositives', 'False\nPositives', 'False\nNegatives']
# counts = [overlap_results['tp'], overlap_results['fp'], overlap_results['fn']]
# colors4 = ['#06FFA5', '#FF6B6B', '#FFD93D']
# bars4 = ax4.bar(categories, counts, color=colors4, alpha=0.8, edgecolor='black')
# ax4.set_ylabel('Count', fontsize=12, fontweight='bold')
# ax4.set_title('Span-level Error Analysis (Overlap)', fontsize=13, fontweight='bold')
# ax4.grid(axis='y', alpha=0.3)
# for bar, count in zip(bars4, counts):
#     height = bar.get_height()
#     ax4.text(bar.get_x() + bar.get_width()/2., height + 50,
#              f'{count:,}', ha='center', va='bottom', fontweight='bold')

# plt.tight_layout()
# plt.show()

# print("✅ Overview plot saved")

In [ ]:
# # 2. Per-technique F1 bar chart
# fig, ax = plt.subplots(figsize=(14, 10))

# tech_df_sorted = tech_df.sort_values('f1', ascending=True)
# techniques = tech_df_sorted['technique'].values
# f1_scores = tech_df_sorted['f1'].values

# # Colour-code bars by F1 range
# colors = []
# for score in f1_scores:
#     if score < 0.3:
#         colors.append('#E63946')  # 红色 - poor
#     elif score < 0.5:
#         colors.append('#F77F00')  # 橙色 - below average
#     elif score < 0.7:
#         colors.append('#FFD93D')  # 黄色 - moderate
#     elif score < 0.9:
#         colors.append('#6BCB77')  # 绿色 - good
#     else:
#         colors.append('#06FFA5')  # 青色 - excellent

# bars = ax.barh(techniques, f1_scores, color=colors, alpha=0.8, edgecolor='black')

# # Value labels
# for bar, score in zip(bars, f1_scores):
#     width = bar.get_width()
#     ax.text(width + 0.01, bar.get_y() + bar.get_height()/2.,
#             f'{score:.3f}', ha='left', va='center', fontweight='bold', fontsize=9)

# ax.set_xlabel('F1 Score', fontsize=13, fontweight='bold')
# ax.set_title('F1 Score by Propaganda Technique', fontsize=15, fontweight='bold', pad=20)
# ax.set_xlim([0, 1.0])
# ax.grid(axis='x', alpha=0.3)

# # Legend
# from matplotlib.patches import Patch
# legend_elements = [
#     Patch(facecolor='#E63946', label='F1 < 0.3 (poor)'),
#     Patch(facecolor='#F77F00', label='0.3 ≤ F1 < 0.5 (below average)'),
#     Patch(facecolor='#FFD93D', label='0.5 ≤ F1 < 0.7 (moderate)'),
#     Patch(facecolor='#6BCB77', label='0.7 ≤ F1 < 0.9 (good)'),
#     Patch(facecolor='#06FFA5', label='F1 ≥ 0.9 (excellent)')
# ]
# ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

# plt.tight_layout()
# plt.show()

# print("✅ Per-technique F1 plot saved")

In [ ]:
# # 3. Precision-Recall scatter plot
# fig, ax = plt.subplots(figsize=(12, 10))

# precisions = tech_df['precision'].values
# recalls = tech_df['recall'].values
# f1s = tech_df['f1'].values
# techniques = tech_df['technique'].values

# # Point size and colour mapped to F1
# sizes = f1s * 500 + 50
# scatter = ax.scatter(precisions, recalls, s=sizes, c=f1s, cmap='RdYlGn', 
#                      alpha=0.7, edgecolors='black', linewidths=1.5)

# # Point labels
# for i, tech in enumerate(techniques):
#     # Label only high/low-F1 techniques
#     if f1s[i] > 0.7 or f1s[i] < 0.3:
#         ax.annotate(tech, (precisions[i], recalls[i]), 
#                    fontsize=8, ha='right', va='bottom',
#                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))

# # Diagonal reference line
# ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='F1 = Precision = Recall')

# ax.set_xlabel('Precision', fontsize=13, fontweight='bold')
# ax.set_ylabel('Recall', fontsize=13, fontweight='bold')
# ax.set_title('Precision vs Recall (point size = F1 score)', fontsize=15, fontweight='bold', pad=20)
# ax.set_xlim([0, 1.05])
# ax.set_ylim([0, 1.05])
# ax.grid(True, alpha=0.3)
# ax.legend(fontsize=10)

# # Colour bar
# cbar = plt.colorbar(scatter, ax=ax)
# cbar.set_label('F1 Score', fontsize=12, fontweight='bold')

# plt.tight_layout()
# plt.show()

# print("✅ Precision-Recall scatter plot saved")

In [ ]:
# # Error analysis plots
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# # False Positives
# top_fp = fp_by_tech.most_common(10)
# fp_techs = [t[0] for t in top_fp]
# fp_counts = [t[1] for t in top_fp]

# bars1 = ax1.barh(fp_techs, fp_counts, color='#FF6B6B', alpha=0.8, edgecolor='black')
# ax1.set_xlabel('Count', fontsize=12, fontweight='bold')
# ax1.set_title('Top 10 False Positives by Technique', fontsize=14, fontweight='bold')
# ax1.grid(axis='x', alpha=0.3)
# for bar, count in zip(bars1, fp_counts):
#     width = bar.get_width()
#     ax1.text(width + 5, bar.get_y() + bar.get_height()/2.,
#              f'{count:,}', ha='left', va='center', fontweight='bold')

# # False Negatives
# top_fn = fn_by_tech.most_common(10)
# fn_techs = [t[0] for t in top_fn]
# fn_counts = [t[1] for t in top_fn]

# bars2 = ax2.barh(fn_techs, fn_counts, color='#FFD93D', alpha=0.8, edgecolor='black')
# ax2.set_xlabel('Count', fontsize=12, fontweight='bold')
# ax2.set_title('Top 10 False Negatives by Technique', fontsize=14, fontweight='bold')
# ax2.grid(axis='x', alpha=0.3)
# for bar, count in zip(bars2, fn_counts):
#     width = bar.get_width()
#     ax2.text(width + 2, bar.get_y() + bar.get_height()/2.,
#              f'{count:,}', ha='left', va='center', fontweight='bold')

# plt.tight_layout()
# plt.show()

# print("✅ Error analysis plot saved")

## 📊 Final summary

In [ ]:
# print("\n" + "="*70)
# print("🎯 最终结果汇总")
# print("="*70)

# print("\n📊 数据统计:")
# print(f"  真实spans总数:  {len(true_spans):,}")
# print(f"  预测spans总数:  {len(pred_spans):,}")
# print(f"  文章总数:       {len(true_articles | pred_articles)}")
# print(f"  技术类型总数:   {len(multilabel_results['all_techniques'])}")

# print("\n📈 Span-level 性能:")
# print(f"  精确匹配:")
# print(f"    Precision: {exact_results['precision']:.4f} ({exact_results['precision']*100:.2f}%)")
# print(f"    Recall:    {exact_results['recall']:.4f} ({exact_results['recall']*100:.2f}%)")
# print(f"    F1 Score:  {exact_results['f1']:.4f} ({exact_results['f1']*100:.2f}%)")
# print(f"  重叠匹配:")
# print(f"    Precision: {overlap_results['precision']:.4f} ({overlap_results['precision']*100:.2f}%)")
# print(f"    Recall:    {overlap_results['recall']:.4f} ({overlap_results['recall']*100:.2f}%)")
# print(f"    F1 Score:  {overlap_results['f1']:.4f} ({overlap_results['f1']*100:.2f}%)")

# print("\n📈 多标签分类性能:")
# print(f"  Macro F1:    {multilabel_results['macro_f1']:.4f} ({multilabel_results['macro_f1']*100:.2f}%)")
# print(f"  Micro F1:    {multilabel_results['micro_f1']:.4f} ({multilabel_results['micro_f1']*100:.2f}%)")
# print(f"  Weighted F1: {multilabel_results['weighted_f1']:.4f} ({multilabel_results['weighted_f1']*100:.2f}%)")

# print("\n📈 二分类性能 (文章级别):")
# print(f"  Accuracy:  {binary_results['accuracy']:.4f} ({binary_results['accuracy']*100:.2f}%)")
# print(f"  Precision: {binary_results['precision']:.4f} ({binary_results['precision']*100:.2f}%)")
# print(f"  Recall:    {binary_results['recall']:.4f} ({binary_results['recall']*100:.2f}%)")
# print(f"  F1 Score:  {binary_results['f1']:.4f} ({binary_results['f1']*100:.2f}%)")

# print("\n🏆 表现最好的技术 (Top 5):")
# top5 = tech_df.head(5)
# for idx, row in top5.iterrows():
#     print(f"  {row['technique']:40s} F1: {row['f1']:.3f}")

# print("\n⚠️ 表现最差的技术 (Bottom 5):")
# bottom5 = tech_df.tail(5).sort_values('f1')
# for idx, row in bottom5.iterrows():
#     print(f"  {row['technique']:40s} F1: {row['f1']:.3f}")

# print("\n" + "="*70)
# print("✅ 评估完成！")
# print("="*70)

## 💾 Export results to CSV

In [ ]:
# # Export detailed results
# output_dir = "your/path/here"
# timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

# # 1. Export per-technique metrics
# tech_output_file = f"{output_dir}po_context_all_ai_F1.tsv"
# tech_df.to_csv(tech_output_file, index=False, encoding='utf-8')
# print(f"✅ 技术详细指标已保存至: {tech_output_file}")

# # 2. Export overall summary
# summary_data = {
#     'Metric': [
#         'Span_Exact_Precision', 'Span_Exact_Recall', 'Span_Exact_F1',
#         'Span_Overlap_Precision', 'Span_Overlap_Recall', 'Span_Overlap_F1',
#         'Multilabel_Macro_F1', 'Multilabel_Micro_F1', 'Multilabel_Weighted_F1',
#         'Binary_Accuracy', 'Binary_Precision', 'Binary_Recall', 'Binary_F1'
#     ],
#     'Score': [
#         exact_results['precision'], exact_results['recall'], exact_results['f1'],
#         overlap_results['precision'], overlap_results['recall'], overlap_results['f1'],
#         multilabel_results['macro_f1'], multilabel_results['micro_f1'], multilabel_results['weighted_f1'],
#         binary_results['accuracy'], binary_results['precision'], binary_results['recall'], binary_results['f1']
#     ]
# }
# summary_df = pd.DataFrame(summary_data)
# summary_output_file = f"{output_dir}overall_summary_{timestamp}.csv"
# summary_df.to_csv(summary_output_file, index=False, encoding='utf-8')
# print(f"✅ 整体性能汇总已保存至: {summary_output_file}")

# print("\n所有结果已成功导出！")

## 📝 Usage notes

### How to use this notebook for other datasets:

1. **Update paths in the Configuration cell**
   - `PREDICTION_FILE`: path to your prediction file
   - `LABEL_FILE`: path to your gold-standard label file

2. **Check file format**
   - If column names differ, update `PRED_COLUMNS`
   - If article_id format differs, update `PRED_HAS_ARTICLE_PREFIX` / `LABEL_HAS_ARTICLE_PREFIX`

3. **Run all cells**
   - Menu: Cell → Run All
   - Or: Shift + Enter to run cell by cell

4. **Inspect results**
   - All metrics are printed in cell output
   - Visualisation plots are generated inline
   - Detailed results are exported as CSV

### Supported metrics:
- ✅ Span-level F1 (exact match & overlap match)
- ✅ Multi-label classification F1 (Macro / Micro / Weighted)
- ✅ Binary classification Accuracy & F1 (article level)
- ✅ Per-technique detailed metrics
- ✅ Error analysis (FP / FN breakdown)

### FAQ:

**Q: My prediction file is CSV, not TSV — what do I change?**  
A: In `load_prediction_tsv`, change `sep='\t'` to `sep=','`

**Q: How do I filter predictions by confidence score?**  
A: Set `CONFIDENCE_THRESHOLD = 0.5` in the Configuration cell

**Q: How do I save the visualisation plots?**  
A: Add `plt.savefig('figure_name.png', dpi=300, bbox_inches='tight')` before each `plt.show()`

---
